# 04 Missing Phenotype Prediction

Этот ноутбук делает **финальное предсказание пропущенных фенотипов**:

1. **Выбирает лучшую модель** по результатам CV (самая высокая средняя корреляция)
2. **Обучает её на всех известных фенотипах** (все данные с non-NA y)
3. **Предсказывает y для записей с NA** — это и есть "genomic predictions" или "breeding values"

**Для кого**: для тех, кто хочет увидеть финальный результат — предсказанные значения для тех особей, у которых нет измеренного фенотипа.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

from gp_py.io import fn_load_genotype, fn_load_phenotype, fn_filter_genotype, fn_filter_phenotype, fn_merge_genotype_and_phenotype
from gp_py.cv import fn_cross_validation_within_population
from gp_py.schema import MergedData

## 1. Подготовка данных (полные, с пропусками)

In [ ]:
DATA_DIR = Path("../inst/exec_Rscript/input")

G = fn_load_genotype(str(DATA_DIR / "test_geno.Rds"))
list_pheno = fn_load_phenotype(str(DATA_DIR / "test_pheno.tsv"))

# Фильтрация
Gf = fn_filter_genotype(G)
phf = fn_filter_phenotype(list_pheno)
merged = fn_merge_genotype_and_phenotype(Gf, phf)

print(f"Общий размер после merge: {merged.G.shape}")

known_mask = merged.y.notna()
missing_mask = merged.y.isna()

print(f"  Известные фенотипы: {known_mask.sum()}")
print(f"  Пропущенные (будут предсказаны): {missing_mask.sum()}")

## 2. Запуск CV для выбора лучшей модели

In [ ]:
# Данные только с известными фенотипами для CV
merged_known = MergedData(
    G=merged.G.loc[known_mask].copy(),
    y=merged.y.loc[known_mask].copy(),
    pop=merged.pop.loc[known_mask].copy(),
    trait_name=merged.trait_name,
)

# CV
cv_results = fn_cross_validation_within_population(
    merged_known,
    n_folds=3,
    n_reps=2,
    vec_models_to_test=("ridge", "lasso", "elastic_net"),
    bool_parallel=False,
    bayes_backend="native",
    verbose=False
)

metrics_df = cv_results["METRICS_WITHIN_POP"]
print("CV результаты:")
print(metrics_df.groupby("model")["corr"].mean().sort_values(ascending=False))

## 3. Выбор лучшей модели

In [ ]:
# Лучшая модель по средней корреляции
model_perf = metrics_df.groupby("model")["corr"].mean()
best_model = model_perf.idxmax()
best_corr = model_perf.max()

print("=== Лучшая модель ===")
print(f"  Модель: {best_model}")
print(f"  Средняя корреляция на CV: {best_corr:.4f}")

## 4. Финальное обучение и предсказание

In [ ]:
from gp_py.models import fn_ridge, fn_lasso, fn_elastic_net

MODEL_MAP = {
    "ridge": fn_ridge,
    "lasso": fn_lasso,
    "elastic_net": fn_elastic_net,
}

fn_best = MODEL_MAP[best_model]

# Индексы: правильные булевы маски
train_idx = list(merged.y.dropna().index)
missing_idx = list(merged.y[merged.y.isna()].index)

print(f"Обучаем {best_model} на {len(train_idx)} записях...")
print(f"Предсказываем для {len(missing_idx)} записей с пропусками...")

# Финальное предсказание
result = fn_best(
    merged,
    vec_idx_training=train_idx,
    vec_idx_validation=missing_idx,
    other_params={"bayes_backend": "native"},
    verbose=False
)

print("Готово!")

## 5. Результаты предсказаний

In [ ]:
pred_df = result["df_y_validation"]
print("=== Предсказанные фенотипы (GENOMIC_PREDICTIONS) ===")
print(f"Число предсказаний: {len(pred_df)}")
print(f"\nПервые 10 записей:")
print(pred_df.head(10).to_string())

print(f"\nСтатистика предсказаний:")
print(f"  Min: {pred_df['y_pred'].min():.4f}")
print(f"  Max: {pred_df['y_pred'].max():.4f}")
print(f"  Mean: {pred_df['y_pred'].mean():.4f}")
print(f"  Std: {pred_df['y_pred'].std():.4f}")

## 6. Визуализация предсказаний

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Гистограмма предсказаний
ax1 = axes[0]
ax1.hist(pred_df["y_pred"], bins=20, color="steelblue", alpha=0.7, edgecolor="black")
ax1.set_xlabel("Predicted phenotype")
ax1.set_ylabel("Count")
ax1.set_title(f"Distribution of predictions ({best_model})")
ax1.grid(True, alpha=0.3)

# По популяции
ax2 = axes[1]
for pop in pred_df["pop"].unique():
    subset = pred_df[pred_df["pop"] == pop]["y_pred"]
    ax2.hist(subset, bins=15, alpha=0.5, label=pop)
ax2.set_xlabel("Predicted phenotype")
ax2.set_ylabel("Count")
ax2.set_title("Predictions by population")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Итог

- Мы выбрали лучшую модель по CV (самая высокая средняя корреляция)
- Обучили её на всех записях с известным фенотипом
- Предсказали значения для записей с пропущенным фенотипом
- Получили **genomic predictions** — оценки breeding values для каждой особи

**Следующий шаг**: перейти к ноутбуку `05_parity_vs_r`, чтобы сравнить результаты Python-пайплайна с оригинальным R-пакетом.